In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import boxcox
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.linear_model import Lasso, LassoCV, ElasticNetCV, ElasticNet
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
import pmdarima as pm

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

import xgboost as xgb
import lightgbm as lgb

import json
import time
import os

sns.set_style('darkgrid')
magma_palette = sns.color_palette('magma', 6)
plt.rcParams['figure.figsize'] = (15, 6)

tf.get_logger().setLevel('ERROR')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)



In [ ]:
df = pd.read_csv('Data/processed_dataset.csv', parse_dates=['Date'], index_col='Date')

print(f"\nDataset shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"Total missing values: {df.isnull().sum().sum()}")

TARGET_COLUMN = 'food_price_index'

EXOGENOUS_FEATURES = [
    'market_price_maize', 'market_price_rice', 'market_price_sorghum',
    'market_price_oil', 'climate_temperature', 'climate_precipitation',
    'climate_humidity', 'population', 'conflict_critical',
    'conflict_heightened', 'conflict_typical', 'drought_ndvi_critical',
    'drought_ndvi_heightened', 'drought_ndvi_typical',
    'drought_rainfall_critical', 'drought_rainfall_heightened',
    'drought_rainfall_typical', 'exchange_rate_critical',
    'exchange_rate_heightened', 'exchange_rate_typical',
    'food_price_critical', 'food_price_heightened', 'food_price_typical',
    'fuel_price_critical', 'fuel_price_heightened', 'fuel_price_typical',
    'risk_population_estimated', 'risk_population_percentage',
    'heavy_rainfall_critical', 'heavy_rainfall_heightened',
    'heavy_rainfall_typical', 'cpi_alcohol_tobacco_narcotics',
    'cpi_clothing_footwear', 'cpi_communication', 'cpi_education',
    'cpi_food_non_alcoholic_beverages', 'cpi_furnishings_household_maintenance',
    'cpi_health', 'cpi_housing_utilities', 'cpi_miscellaneous',
    'cpi_recreation_culture', 'cpi_restaurants_hotels', 'cpi_transport',
    'exchange_rate', 'gdp_per_capita_usd', 'government_final_consumption',
    'gross_fixed_capital_formation', 'household_final_consumption',
    'imports_goods_services', 'month', 'year', 'quarter'
]

print(f"\nTarget variable: {TARGET_COLUMN}")
print(f"Number of features: {len(EXOGENOUS_FEATURES)}")

print("\nMissing values in features:")
missing_counts = df[EXOGENOUS_FEATURES].isnull().sum()
missing_features = missing_counts[missing_counts > 0]
if len(missing_features) > 0:
    for feat, count in missing_features.items():
        print(f"  - {feat}: {count} ({count/len(df)*100:.2f}%)")
else:
    print("  No missing values found")


In [ ]:
df_clean = df.dropna()
df_clean = df_clean.sort_index()
print(f"\nCleaned dataset shape (after dropping missing values): {df_clean.shape}")

In [ ]:
train_size = int(len(df_clean) * 0.90)
train_df = df_clean.iloc[:train_size].copy()
test_df = df_clean.iloc[train_size:].copy()

print(f"\nTraining set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}")
print(f"Train date range: {train_df.index.min()} to {train_df.index.max()}")
print(f"Test date range: {test_df.index.min()} to {test_df.index.max()}")

In [ ]:
def add_lags(data, target_col, lags=[1, 3, 6]):
    for lag in lags:
        data[f'{target_col}_lag_{lag}'] = data[target_col].shift(lag)
    data[f'{target_col}_rolling_mean_3'] = data[target_col].rolling(window=3).mean()
    return data

train_df = add_lags(train_df, TARGET_COLUMN)
test_df = add_lags(test_df, TARGET_COLUMN)

train_df.dropna(inplace=True)
test_df.dropna(inplace=True)

LAG_FEATURES = [f'{TARGET_COLUMN}_lag_{l}' for l in [1, 3, 6]] + [f'{TARGET_COLUMN}_rolling_mean_3']
ALL_FEATURES = EXOGENOUS_FEATURES + LAG_FEATURES

In [ ]:
# Transformation and Scaling
y_train_raw = train_df[TARGET_COLUMN]
y_train, lambda_boxcox = boxcox(y_train_raw)
train_df['target_transformed'] = y_train

test_df['target_transformed'] = stats.boxcox(test_df[TARGET_COLUMN], lmbda=lambda_boxcox)
y_test = test_df['target_transformed']


region_le = LabelEncoder()
district_le = LabelEncoder()

train_df['region'] = region_le.fit_transform(train_df['region'])
train_df['district'] = district_le.fit_transform(train_df['district'])

test_df['region'] = test_df['region'].map(lambda s: region_le.transform([s])[0] if s in region_le.classes_ else -1)
test_df['district'] = test_df['district'].map(lambda s: district_le.transform([s])[0] if s in district_le.classes_ else -1)

scaler = RobustScaler()
train_df[ALL_FEATURES] = scaler.fit_transform(train_df[ALL_FEATURES])
test_df[ALL_FEATURES] = scaler.transform(test_df[ALL_FEATURES])

X_train = train_df[ALL_FEATURES]
X_test = test_df[ALL_FEATURES]


In [ ]:
print("\nPerforming feature selection with Elastic Net...")

tscv = TimeSeriesSplit(n_splits=5)

elastic_cv = ElasticNetCV(
    l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 1.0],  # 1.0 ≈ Lasso
    alphas=None,                            # let sklearn choose grid
    cv=tscv,
    random_state=RANDOM_STATE,
    max_iter=10000,
    n_jobs=-1
)

elastic_cv.fit(X_train, y_train)

optimal_alpha = elastic_cv.alpha_
optimal_l1_ratio = elastic_cv.l1_ratio_

print(f"Optimal Elastic Net alpha: {optimal_alpha:.6f}")
print(f"Optimal Elastic Net l1_ratio: {optimal_l1_ratio:.2f}")

elastic = ElasticNet(
    alpha=optimal_alpha,
    l1_ratio=optimal_l1_ratio,
    max_iter=10000,
    random_state=RANDOM_STATE
)

elastic.fit(X_train, y_train)

selected_features = X_train.columns[elastic.coef_ != 0].tolist()


for col in ['region', 'district']:
    if col not in selected_features:
        selected_features.insert(0, col)


print(f"\nSelected {len(selected_features)} features out of {len(EXOGENOUS_FEATURES)}:")


In [ ]:
for i, feat in enumerate(selected_features, 1):
    coef = elastic.coef_[X_train.columns.get_loc(feat)]
    print(f"  {i}. {feat}: {coef:.4f}")

scaler = RobustScaler()

X_train = pd.DataFrame(
    scaler.fit_transform(X_train[selected_features]),
    columns=selected_features,
    index=X_train.index
)

X_test = pd.DataFrame(
    scaler.transform(X_test[selected_features]),
    columns=selected_features,
    index=X_test.index
)

In [ ]:
N_SPLITS = 5
TIMESTEPS = 12  # Lookback for LSTM/GRU

tscv = TimeSeriesSplit(n_splits=N_SPLITS)

all_metrics = {}
all_predictions = {}
training_times = {}

In [ ]:
def inverse_boxcox(y, lambda_val):
    if lambda_val == 0: return np.exp(y)
    return np.power(lambda_val * y + 1, 1/lambda_val)


def evaluate_model(y_true_transformed, y_pred_transformed, name, y_train_pred=None, y_train_true=None):
    y_true = inverse_boxcox(y_true_transformed, lambda_boxcox)
    y_pred = inverse_boxcox(y_pred_transformed, lambda_boxcox)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    overfitting_coeff = np.nan
    if y_train_pred is not None and y_train_true is not None:
        y_train_true_inv = inverse_boxcox(y_train_true, lambda_boxcox)
        y_train_pred_inv = inverse_boxcox(y_train_pred, lambda_boxcox)
        rmse_train = np.sqrt(mean_squared_error(y_train_true_inv, y_train_pred_inv))
        overfitting_coeff = (rmse - rmse_train) / rmse_train if rmse_train != 0 else 0

    print(f"{name} - MAE: {mae:.4f}, RMSE: {rmse:.4f}, R2: {r2:.4f}, Overfitting: {overfitting_coeff:.4f}")

    metrics = {
        'model': name,
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2,
        'Overfitting_Coeff': overfitting_coeff
    }

    all_metrics[name] = metrics
    all_predictions[name] = y_pred.tolist()
    
    return metrics

def create_sequences(X, y, timesteps):
    Xs, ys = [], []
    for i in range(len(X) - timesteps):
        Xs.append(X.iloc[i:(i + timesteps)].values)
        ys.append(y[i + timesteps])
    return np.array(Xs), np.array(ys)

X_train_seq, y_train_seq = create_sequences(X_train, train_df['target_transformed'], TIMESTEPS)
X_test_seq, y_test_seq = create_sequences(X_test, test_df['target_transformed'], TIMESTEPS)

    

In [ ]:
from tensorflow.keras import regularizers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


def build_lstm_base(
    n_features,
    units=64,
    dropout_rate=0.0,
    l2_reg=0.0
):
    reg = regularizers.l2(l2_reg) if l2_reg > 0 else None

    model = Sequential([
        LSTM(
            units,
            activation="tanh",
            input_shape=(1, n_features),
            dropout=dropout_rate,
            recurrent_dropout=dropout_rate,
            kernel_regularizer=reg,
            recurrent_regularizer=reg,
            bias_regularizer=reg
        ),
        Dense(1, kernel_regularizer=reg)
    ])

    model.compile(
        optimizer=Adam(learning_rate=1e-3, clipnorm=1.0),
        loss="mse"
    )

    return model

def build_gru_residual(
    n_features,
    units=8,
    dropout_rate=0.1,
    l2_reg=0.0
):
    reg = regularizers.l2(l2_reg) if l2_reg > 0 else None

    model = Sequential([
        GRU(
            units,
            activation="tanh",
            input_shape=(1, n_features),
            dropout=dropout_rate,
            recurrent_dropout=dropout_rate,
            kernel_regularizer=reg,
            recurrent_regularizer=reg,
            bias_regularizer=reg
        ),
        Dense(1, kernel_regularizer=reg)
    ])

    model.compile(
        optimizer=Adam(learning_rate=1e-3, clipnorm=1.0),
        loss="mse"
    )

    return model


def get_callbacks(use_early_stopping):
    callbacks = [
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=5,
            min_lr=1e-6
        )
    ]

    if use_early_stopping:
        callbacks.insert(
            0,
            EarlyStopping(
                monitor="val_loss",
                patience=3,
                restore_best_weights=True
            )
        )

    return callbacks

experiments = {
    "A_baseline": dict(
        lstm_units=64,
        gru_units=32,
        dropout=0.0,
        l2=0.0,
        early_stopping=False
    ),

    # "B_dropout": dict(
    #     lstm_units=64,
    #     gru_units=32,
    #     dropout=0.3,
    #     l2=0.0,
    #     early_stopping=False
    # ),

    "C_l2": dict(
        lstm_units=64,
        gru_units=32,
        dropout=0.0,
        l2=1e-4,
        early_stopping=False
    ),

    # "D_smaller": dict(
    #     lstm_units=32,
    #     gru_units=16,
    #     dropout=0.0,
    #     l2=0.0,
    #     early_stopping=False
    # ),

    # "E_dropout_l2": dict(
    #     lstm_units=64,
    #     gru_units=32,
    #     dropout=0.3,
    #     l2=1e-4,
    #     early_stopping=False
    # ),

    # "F_full_regularized": dict(
    #     lstm_units=32,
    #     gru_units=16,
    #     dropout=0.3,
    #     l2=1e-4,
    #     early_stopping=True
    # ),
}


In [ ]:
X_train_hybrid = X_train.values.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test_hybrid  = X_test.values.reshape((X_test.shape[0], 1, X_test.shape[1]))

# Optional: split validation set
from sklearn.model_selection import train_test_split

X_train_hybrid, X_val_hybrid, y_train_split, y_val = train_test_split(
    X_train_hybrid, y_train, test_size=0.2, random_state=RANDOM_STATE, shuffle=False
)

In [ ]:

histories = {}
final_metrics = {}

for name, cfg in experiments.items():
    print(f"\nTraining {name}")

    # --- LSTM base ---
    lstm_model = build_lstm_base(
        n_features=X_train.shape[1],
        units=cfg["lstm_units"],
        dropout_rate=cfg["dropout"],
        l2_reg=cfg["l2"]
    )

    lstm_history = lstm_model.fit(
        X_train_hybrid,
        y_train_split,
        validation_data=(X_val_hybrid, y_val),
        epochs=200,
        batch_size=32,
        callbacks=get_callbacks(cfg["early_stopping"]),
        verbose=0
    )

    # --- Residuals ---
    lstm_train_pred = lstm_model.predict(
        X_train_hybrid, verbose=0
    ).flatten()

    train_residuals = y_train_split - lstm_train_pred

    # --- GRU residual model ---
    gru_model = build_gru_residual(
        n_features=X_train.shape[1],
        units=cfg["gru_units"],
        dropout_rate=cfg["dropout"],
        l2_reg=cfg["l2"]
    )

    gru_history = gru_model.fit(
        X_train_hybrid,
        train_residuals,
        validation_data=(X_val_hybrid, y_val - lstm_model.predict(X_val_hybrid, verbose=0).flatten()),
        epochs=200,
        batch_size=32,
        callbacks=get_callbacks(cfg["early_stopping"]),
        verbose=0
    )

    # --- Store learning curves (combined) ---
    histories[name] = {
        "lstm": lstm_history.history,
        "gru": gru_history.history
    }

    # --- Final hybrid predictions ---
    train_hybrid_pred = (
        lstm_train_pred +
        gru_model.predict(X_train_hybrid, verbose=0).flatten()
        )

    val_hybrid_pred = (
        lstm_model.predict(X_val_hybrid, verbose=0).flatten() +
        gru_model.predict(X_val_hybrid, verbose=0).flatten()
    )

    final_metrics[name] = {
        "train_loss": ((y_train_split - train_hybrid_pred) ** 2).mean(),
        "val_loss": ((y_val - val_hybrid_pred) ** 2).mean(),
        "gap": ((y_val - val_hybrid_pred) ** 2).mean()
            - ((y_train_split - train_hybrid_pred) ** 2).mean()
    }





In [ ]:
import os
import joblib

# os.makedirs("Models", exist_ok=True)

# joblib.dump(best_gb, "Models/best_gb_model.pkl")
# print("Gradient Boosting model saved to Models/best_gb_model.pkl")

# lstm_model.save("Models/lstm_model.h5")
# gru_model.save("Models/gru_model.h5")
# print("LSTM and GRU models saved to Models/")

lstm_model.save("Models/lstm_model_regularized.h5")
gru_model.save("Models/gru_model_regularized.h5")
print("LSTM and GRU regularized models saved to Models/")

# feature_names = X_train.columns.tolist()
# joblib.dump(feature_names, "Models/feature_names.pkl")
# print("Feature names saved to Models/feature_names.pkl")

# joblib.dump(scaler, "Models/scaler.pkl")
# print("RobustScaler saved to Models/scaler.pkl")

# joblib.dump(lambda_boxcox, "Models/lambda_boxcox.pkl")
# print("Box-Cox lambda saved to Models/lambda_boxcox.pkl")

# joblib.dump(region_le, "Models/region_encoder.pkl")
# joblib.dump(district_le, "Models/district_encoder.pkl")
# print("Region and district encoders saved to Models/")
